In [1]:
"""
Reproduces Fig. 2, Fig. 3 and Fig. 4 from Ratnakar, R.R. (2026), 
"Thermodynamics and equilibrium thermochemistry of ortho- and 
para-hydrogen: Integrating quantum mechanics with classical
EOS modeling", Int. J. Hydrogen Energy 245, 155702.
https://doi.org/10.1016/j.ijhydene.2026.155702.
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
from h2_thermo_final import R, ideal_properties, Q_partition

OUTDIR = "figures"

# ----------------------------------------------------------------------------------
# Common temperature grids (paper uses ~0-600K linear for Fig.2-3,
# log-scale ~10-500K for Fig.4)
# ----------------------------------------------------------------------------------
T_lin = np.linspace(5, 600, 400)          # for Fig.2, Fig.3
T_log = np.geomspace(10, 500, 300)        # for Fig.4

species_list = ["p", "o", "eq", "n"]
species_label = {"p": "pH2", "o": "oH2", "eq": "eqH2", "n": "nH2"}

# Standard-state ideal-gas molar volume (1 bar, given T). Only used for the
# ABSOLUTE plots; the DIFFERENCE plots are independent of this choice.
P0 = 1.0e5  # [Pa]

def V_std(T):
    return R * T / P0

# ----------------------------------------------------------------------------------
# Compute ideal-state properties for every species, over T_lin
# ----------------------------------------------------------------------------------
props = {sp: {"H": [], "S": [], "G": [], "U": [], "Cp": []} for sp in species_list}
for T in T_lin:
    V = V_std(T)
    for sp in species_list:
        d = ideal_properties(T, V, sp)
        for key in ("H", "S", "G", "U", "Cp"):
            props[sp][key].append(d[key])
for sp in species_list:
    for key in props[sp]:
        props[sp][key] = np.array(props[sp][key])


# =======================================================================================================
# FIGURE 1: absolute values H0/R, S0/R, G0/R, U0/R vs T
# =======================================================================================================
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
panel_info = [
    ("H", r"$H^{0}/R$, K", axes[0, 0], "(a)"),
    ("S", r"$S^{0}/R$",    axes[0, 1], "(b)"),
    ("G", r"$G^{0}/R$, K", axes[1, 0], "(c)"),
    ("U", r"$U^{0}/R$, K", axes[1, 1], "(d)"),
]
colors = {"p": "#FF2D2D", "o": "#0066FF", "eq": "#00C853", "n": "#111111"}
for key, ylabel, ax, tag in panel_info:
    for sp in species_list:
        ax.plot(T_lin, props[sp][key] / R, label=species_label[sp], color=colors[sp])
        ax.xaxis.set_major_locator(MultipleLocator(100))
        ax.xaxis.set_minor_locator(MultipleLocator(20))
        ax.yaxis.set_minor_locator(AutoMinorLocator(5))
        ax.tick_params(axis="both", which="major", direction="in", length=3.6)
        ax.tick_params(axis="both", which="minor", direction="in", length=2.2)
    ax.set_xlabel("T, K")
    ax.set_ylabel(ylabel)
    ax.text(0.5, -0.17, tag, transform=ax.transAxes, ha="center", fontweight="bold")  
    ax.legend(loc="upper right", fontsize=8, labelspacing=0.2, handlelength=1.5, borderpad=0.3)
    ax.grid(alpha=0.3)
fig.suptitle(r"Ideal-state ABSOLUTE properties vs T" "\n"
    r"$H_{\mathbf{ref}}=S_{\mathbf{ref}}=U_{\mathbf{ref}}=G_{\mathbf{ref}}=0$ — offset is arbitrary",
    fontsize=12, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(f"{OUTDIR}/fig1_absolute_ideal_properties.png", dpi=150)
plt.close(fig)


# =======================================================================================================
# FIGURE 2: differences delta-H0,S0,G0,U0 vs T (reproduces Fig. 2 of the paper)
# =======================================================================================================
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
diff_info = [
    ("H", r"$\Delta H^{0}/R$, K", axes[0, 0], "(a)"),
    ("S", r"$\Delta S^{0}/R$",    axes[0, 1], "(b)"),
    ("G", r"$\Delta G^{0}/R$, K", axes[1, 0], "(c)"),
    ("U", r"$\Delta U^{0}/R$, K", axes[1, 1], "(d)"),
]
diff_pairs = [
    ("o", "p", "#0066FF"),
    ("n", "p", "#111111"),
    ("eq","p", "#00C853"),
    ("n", "eq","#FF2D2D"),
]
for key, ylabel, ax, tag in diff_info:
    for sp1, sp2, color in diff_pairs:
        label = rf"${key}^{{0}}_{{\mathrm{{{sp1}}}}} - {key}^{{0}}_{{\mathrm{{{sp2}}}}}$"
        ax.plot(T_lin, (props[sp1][key] - props[sp2][key]) / R, label=label, color=color)
        ax.xaxis.set_major_locator(MultipleLocator(100))
        ax.xaxis.set_minor_locator(MultipleLocator(20))
        ax.yaxis.set_minor_locator(AutoMinorLocator(5))
        ax.tick_params(axis="both", which="major", direction="in", length=3.6)
        ax.tick_params(axis="both", which="minor", direction="in", length=2.2)
    ax.set_xlabel("T, K")
    ax.set_ylabel(ylabel)
    ax.text(0.5, -0.17, tag, transform=ax.transAxes, ha="center", fontweight="bold")  
    ax.legend(loc="upper right", fontsize=8, labelspacing=0.2, handlelength=1.5, borderpad=0.3)
    ax.grid(alpha=0.3)
fig.suptitle("Ideal-state property differences between hydrogen isomers vs T\n"
             "(reproduces Fig. 2 of the paper)", fontsize=12, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(f"{OUTDIR}/fig2_differences_ideal_properties.png", dpi=150)
plt.close(fig)


# =======================================================================================================
# FIGURE 3: Cp0/R vs T (reproduces Fig. 3 of the paper)
# =======================================================================================================
fig, ax = plt.subplots(figsize=(7, 5.5))
for sp, style in [("o", "#0066FF"), ("p", "#FF2D2D"), ("n", "#111111"), ("eq", "#00C853")]:
    ax.plot(T_lin, props[sp]["Cp"] / R, label=species_label[sp], color=style)
    ax.xaxis.set_major_locator(MultipleLocator(100))
    ax.xaxis.set_minor_locator(MultipleLocator(20))
    ax.yaxis.set_minor_locator(AutoMinorLocator(5))
    ax.tick_params(axis="both", which="major", direction="in", length=3.6)
    ax.tick_params(axis="both", which="minor", direction="in", length=2.2)
ax.axhline(2.5, color="gray", ls=":", lw=1, label="5/2 R (low-T limit)")
ax.axhline(3.5, color="gray", ls="--", lw=1, label="7/2 R (high-T limit)")
ax.set_xlabel("T, K")
ax.set_ylabel(r"$C_{\mathrm{P}}^0/R$")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.suptitle("Ideal-state isobaric heat capacity vs T\n"
             "(reproduces Fig. 3 of the paper)", fontsize=12, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(f"{OUTDIR}/fig3_Cp0_ideal.png", dpi=150)
plt.close(fig)


# =======================================================================================================
# FIGURE 4: equilibrium para-H2 mole fraction & heat of conversion vs T (reproduces Fig. 4 of the paper)
# =======================================================================================================
y_eq = np.array([Q_partition(T, "p") / (Q_partition(T, "p") + Q_partition(T, "o")) for T in T_log])
Hn_minus_Hp = np.array([
    (0.25*ideal_properties(T, 1.0, "p")["H"] + 0.75*ideal_properties(T, 1.0, "o")["H"])
    - ideal_properties(T, 1.0, "p")["H"]
    for T in T_log
])

fig, ax1 = plt.subplots(figsize=(7, 5.5))
ax1.plot(T_log, y_eq, color="#FF2D2D", label=r"$y_{eq}$ (para mole fraction)")
ax1.set_xscale("log")
ax1.set_xlabel("T, K")
ax1.set_ylabel(r"$y_{\mathrm{eq}}$", color="#FF2D2D")
ax1.axhline(0.25, color="#FF2D2D", ls=":", lw=1)
ax1.axhline(1.0, color="#FF2D2D", ls=":", lw=1)
ax1.set_ylim(-0.05, 1.1)
ax1.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax1.yaxis.set_minor_locator(AutoMinorLocator(5))
ax1.tick_params(axis="y", which="major", labelcolor="#FF2D2D", direction="in", length=3.6)
ax1.tick_params(axis="y", which="minor", labelcolor="#FF2D2D", direction="in", length=2.2)
ax1.grid(alpha=0.3)
ax1.tick_params(axis="x", which="major", direction="in", length=3.6)
ax1.tick_params(axis="x", which="minor", direction="in", length=2.2)

ax2 = ax1.twinx()
ax2.plot(T_log, Hn_minus_Hp, color="#0066FF", label=r"$H_{\mathrm{n}}^0-H_{\mathrm{p}}^0$")
ax2.set_ylabel(r"$H_{\mathrm{n}}^0 - H_{\mathrm{p}}^0$, $\mathrm{J}\cdot\mathrm{mol}^{-1}$", 
               color="#0066FF")
ax2.set_ylim(-50, 1100)
ax2.set_yticks([0, 200, 400, 600, 800, 1000])
ax2.yaxis.set_minor_locator(AutoMinorLocator(5))
ax2.tick_params(axis="y", which="major", labelcolor="#0066FF", direction="in", length=3.6)
ax2.tick_params(axis="y", which="minor", labelcolor="#0066FF", direction="in", length=2.2)

ax1.set_title("Equilibrium para-H2 fraction and heat of conversion vs T\n"
              "(reproduces Fig. 4 of the paper)", fontsize=12, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(f"{OUTDIR}/fig4_equilibrium_and_heat_of_conversion.png", dpi=150)
plt.close(fig)


print("Done. Saved:")
print("  - fig1_absolute_ideal_properties.png           (H0,S0,G0,U0 vs T — ARBITRARY vertical offset)")
print("  - fig2_differences_ideal_properties.png        (Delta-H0,S0,G0,U0 vs T — matches Fig.2 of the paper)")
print("  - fig3_Cp0_ideal.png                           (Cp0/R vs T — matches Fig.3 of the paper)")
print("  - fig4_equilibrium_and_heat_of_conversion.png  (y_eq and Hn-Hp vs T — matches Fig.4 of the paper")

Done. Saved:
  - fig1_absolute_ideal_properties.png           (H0,S0,G0,U0 vs T — ARBITRARY vertical offset)
  - fig2_differences_ideal_properties.png        (Delta-H0,S0,G0,U0 vs T — matches Fig.2 of the paper)
  - fig3_Cp0_ideal.png                           (Cp0/R vs T — matches Fig.3 of the paper)
  - fig4_equilibrium_and_heat_of_conversion.png  (y_eq and Hn-Hp vs T — matches Fig.4 of the paper
